# <font color="#418FDE" size="6.5" uppercase>**Capstone Tool Design**</font>

>Last update: 20260608.
    
By the end of this Lecture, you will be able to:
- Design a command-line engineering tool with clear inputs, calculations, outputs, and error behavior. 
- Integrate functions, collections, file handling, and formatted reports into one coherent script. 
- Evaluate the final tool for correctness, readability, and usefulness in an electrical engineering workflow. 


## **1. Project Planning**

### **1.1. Tool Requirements**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_01_01.jpg?v=1780966886" width="250">



>* Define the tool’s engineering purpose clearly
>* Set boundaries for users and decisions

>* Define predictable user interaction and assumptions
>* Communicate inputs, units, and results clearly

>* Plan clear outputs for review and decisions
>* Handle invalid inputs with helpful errors



In [ ]:
#@title Python Code - Tool Requirements

# This script models early tool requirements.
# It uses a voltage drop example.
# Requirements guide inputs, outputs, and errors.

from pathlib import Path

# Store requirement choices in one dictionary.
requirements = {
    "task": "Estimate single-phase feeder voltage drop",
    "input_style": "file plus safe defaults",
    "output_style": "screen summary and text report",
    "error_behavior": "explain invalid engineering values",
}

# Define sample command-line style input data.
sample_job = {
    "voltage_v": 240.0,
    "current_a": 28.0,
    "length_ft": 85.0,
    "resistance_ohm_per_1000ft": 1.24,
}

# Validate values before calculations begin.
def validate_job(job):
    required_keys = ["voltage_v", "current_a", "length_ft", "resistance_ohm_per_1000ft"]
    missing_keys = [key for key in required_keys if key not in job]
    if missing_keys:
        return False, "Missing fields: " + ", ".join(missing_keys)

    numeric_values = [job[key] for key in required_keys]
    if any(value <= 0 for value in numeric_values):
        return False, "All electrical input values must be positive."

    return True, "Inputs passed basic engineering checks."

# Calculate voltage drop using conductor resistance.
def calculate_drop(job):
    loop_length_ft = job["length_ft"] * 2
    conductor_ohms = job["resistance_ohm_per_1000ft"] * loop_length_ft / 1000
    drop_v = job["current_a"] * conductor_ohms
    percent_drop = 100 * drop_v / job["voltage_v"]
    return drop_v, percent_drop

# Format a short report for documentation.
def build_report(job, drop_v, percent_drop):
    status = "PASS" if percent_drop <= 3.0 else "REVIEW"
    lines = [
        "Voltage Drop Tool Requirement Check",
        f"Purpose: {requirements['task']}",
        f"Inputs: {job['voltage_v']} V, {job['current_a']} A, {job['length_ft']} ft",
        f"Result: {drop_v:.2f} V drop, {percent_drop:.2f}%",
        f"Decision: {status} against a 3.00% planning limit",
    ]
    return "\n".join(lines)

# Run validation before producing results.
valid, message = validate_job(sample_job)

# Stop safely if inputs are not usable.
if not valid:
    print("Requirement error:", message)
else:
    drop_v, percent_drop = calculate_drop(sample_job)
    report_text = build_report(sample_job, drop_v, percent_drop)
    Path("voltage_drop_requirement_report.txt").write_text(report_text)
    print(report_text)
    print("Report file: voltage_drop_requirement_report.txt")



### **1.2. Functional Requirements**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_01_02.jpg?v=1780966884" width="250">



>* Define the tool’s required user actions
>* Specify calculations, checks, files, and reports

>* Define user actions, conditions, and expected results
>* Separate core features from future enhancements

>* Plan decisions, limits, and result warnings
>* Define behaviors to support reliable testing



In [ ]:
#@title Python Code - Functional Requirements

# Plan functional requirements before writing engineering tools.
# Each requirement becomes testable tool behavior.
# This example uses transformer loading safely.

# Store realistic command line style inputs.
tool_inputs = dict(
    kva_rating=75.0,
    secondary_voltage=480.0,
    measured_current=82.0,
)

# Define allowable loading requirements clearly.
limits = dict(
    warning_percent=80.0,
    overload_percent=100.0,
    output_file="transformer_report.txt",
)

# Calculate transformer percent loading from inputs.
def calculate_loading(kva_rating, voltage, current):
    apparent_kva = voltage * current * 1.732 / 1000
    percent_load = apparent_kva / kva_rating * 100
    return apparent_kva, percent_load

# Classify results using planned engineering limits.
def classify_loading(percent_load, warning_percent, overload_percent):
    if percent_load >= overload_percent:
        return "OVERLOAD: reduce load or upsize transformer"
    if percent_load >= warning_percent:
        return "WARNING: loading is high"
    return "OK: loading is within planned limit"

# Validate important inputs before calculations.
def validate_inputs(values):
    required = ["kva_rating", "secondary_voltage", "measured_current"]
    missing = [name for name in required if name not in values]
    if missing:
        raise ValueError("Missing required input values")
    if min(values.values()) <= 0:
        raise ValueError("All numeric inputs must be positive")

# Build the formatted engineering report text.
def build_report(values, limits):
    kva, percent = calculate_loading(
        values["kva_rating"], values["secondary_voltage"], values["measured_current"]
    )
    status = classify_loading(
        percent, limits["warning_percent"], limits["overload_percent"]
    )
    return [
        "Transformer loading requirement demo",
        f"Input rating: {values['kva_rating']:.1f} kVA",
        f"Measured current: {values['measured_current']:.1f} A",
        f"Calculated load: {kva:.1f} kVA ({percent:.1f}%)",
        f"Decision output: {status}",
    ]

# Run validation, calculation, reporting, and file handling.
try:
    validate_inputs(tool_inputs)
    report_lines = build_report(tool_inputs, limits)
    report_text = "\n".join(report_lines)
    with open(limits["output_file"], "w") as report_file:
        report_file.write(report_text)
except ValueError as error:
    report_text = f"Tool stopped safely: {error}"

# Print concise output that matches functional requirements.
print(report_text)
print(f"Report file saved: {limits['output_file']}")



### **1.3. Input Requirements**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_01_03.jpg?v=1780966888" width="250">



>* Define required values, units, and ranges
>* Clear inputs prevent confusion and assumptions

>* Choose interactive, argument, or file inputs
>* Match input method to workflow needs

>* Validate inputs and flag unusual values
>* Define defaults, rejections, and error messages



In [ ]:
#@title Python Code - Input Requirements

# Plan inputs before calculations begin.
# Validate engineering values before use.
# Report clear errors and warnings.

# Store simulated command line inputs.
raw_inputs = {
    "voltage_v": "120",
    "current_a": "18",

    "length_ft": "250",
    "material": "copper",
    "phase": "single",
}

# Define accepted materials and resistances.
resistance_table = {
    "copper": 0.00124,
    "aluminum": 0.00204,
}

# Convert required numeric text values.
def read_positive_float(inputs, key, unit):
    text_value = inputs.get(key, "")
    number_value = float(text_value)

    if number_value <= 0:
        raise ValueError(f"{key} must be positive {unit}.")
    return number_value

# Validate categorical engineering choices.
def read_choice(inputs, key, allowed):
    choice_value = inputs.get(key, "").strip().lower()
    if choice_value not in allowed:

        allowed_text = ", ".join(sorted(allowed))
        raise ValueError(f"{key} must be {allowed_text}.")
    return choice_value

# Calculate voltage drop from validated inputs.
def voltage_drop_report(inputs):
    voltage = read_positive_float(inputs, "voltage_v", "volts")
    current = read_positive_float(inputs, "current_a", "amps")

    length = read_positive_float(inputs, "length_ft", "feet")
    material = read_choice(inputs, "material", resistance_table)
    phase = read_choice(inputs, "phase", {"single", "three"})

    multiplier = 2 if phase == "single" else 1.732
    resistance = resistance_table[material]
    drop = multiplier * current * length * resistance

    percent = 100 * drop / voltage
    warning = "OK" if percent <= 3 else "WARNING"
    return voltage, current, length, material, percent, warning

# Run the tool with planned inputs.
try:
    result = voltage_drop_report(raw_inputs)
    voltage, current, length, material, percent, warning = result

    print("Input requirements demo for voltage drop.")
    print(f"Voltage: {voltage:.0f} V, current: {current:.1f} A.")
    print(f"Length: {length:.0f} ft, material: {material}.")

    print(f"Voltage drop: {percent:.2f}% of source voltage.")
    print(f"Engineering status: {warning}.")
except ValueError as error:
    print(f"Input error: {error}")



## **2. Integrated Script Workflow**

### **2.1. Reusable Script Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_02_01.jpg?v=1780966879" width="250">



>* Break engineering scripts into clear functions
>* Improve testing, updates, and understanding

>* Use collections for real engineering datasets
>* Design predictable, reusable processing components

>* Separate file handling from calculations
>* Use report functions for clearer maintenance



In [ ]:
#@title Python Code - Reusable Script Components

# Build reusable parts for engineering automation.
# Keep calculations separate from file handling.
# Produce a short formatted design report.

import csv
from pathlib import Path
from tempfile import TemporaryDirectory

VOLTAGE = 240.0
LIMIT_PERCENT = 3.0
OHMS_PER_1000FT = {"cu_12": 1.93, "cu_10": 1.21, "cu_8": 0.764}

# Create a tiny cable schedule file.
def write_sample_file(folder_path):
    file_path = Path(folder_path) / "cable_schedule.csv"
    rows = [
        ["circuit", "length_ft", "current_a", "wire"],
        ["PANEL-A", "85", "18", "cu_12"],
        ["MOTOR-1", "140", "22", "cu_10"],
        ["PUMP-2", "210", "31", "cu_8"],
    ]

    with file_path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerows(rows)
    return file_path

# Load records from the input file.
def load_circuits(file_path):
    with Path(file_path).open(newline="") as handle:
        reader = csv.DictReader(handle)
        return list(reader)

# Validate and convert raw text records.
def parse_circuit(row):
    circuit = row.get("circuit", "").strip()
    wire = row.get("wire", "").strip()
    length_ft = float(row.get("length_ft", "0"))

    current_a = float(row.get("current_a", "0"))
    if circuit == "" or wire not in OHMS_PER_1000FT:
        raise ValueError("Circuit record has invalid identifying data")
    if length_ft <= 0 or current_a <= 0:
        raise ValueError("Circuit record has invalid numeric data")

    return {
        "circuit": circuit,
        "wire": wire,
        "length_ft": length_ft,
        "current_a": current_a,
    }

# Calculate round-trip voltage drop.
def calculate_drop(record):
    resistance = OHMS_PER_1000FT[record["wire"]]
    drop_v = 2 * record["length_ft"] * resistance * record["current_a"] / 1000
    percent = 100 * drop_v / VOLTAGE

    result = dict(record)
    result["drop_v"] = drop_v
    result["drop_percent"] = percent
    result["status"] = "PASS" if percent <= LIMIT_PERCENT else "CHECK"
    return result

# Format calculated records into report lines.
def make_report(results):
    lines = ["Reusable Component Cable Report"]
    lines.append("Circuit     Wire   Drop(V)   Drop(%)  Status")
    for item in results:
        line = f"{item['circuit']:<10} {item['wire']:<5} {item['drop_v']:>7.2f} {item['drop_percent']:>8.2f}  {item['status']}"
        lines.append(line)

    checked = sum(1 for item in results if item["status"] == "CHECK")
    lines.append(f"Summary: {len(results)} circuits processed, {checked} need review.")
    return "\n".join(lines)

# Run the workflow using separate components.
with TemporaryDirectory() as folder:
    path = write_sample_file(folder)
    raw_records = load_circuits(path)
    parsed = [parse_circuit(row) for row in raw_records]

    results = [calculate_drop(record) for record in parsed]
    report = make_report(results)
    print(report)



### **2.2. Coherent Script Assembly**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_02_02.jpg?v=1780966881" width="250">



>* Build one clear start-to-finish workflow
>* Use reusable functions for detailed calculations

>* Separate file handling from calculation logic
>* Organize data for clear, testable workflows

>* Guide users with clear control flow
>* Handle errors early and format results



In [ ]:
#@title Python Code - Coherent Script Assembly

# Assemble a complete engineering workflow.
# Separate input, calculations, and reporting.
# Keep outputs concise and reviewable.

from pathlib import Path
import csv

# Define a small input file.
input_path = Path("feeder_runs.csv")
report_path = Path("voltage_drop_report.txt")

# Store example feeder records.
rows = [
    {"name": "Feeder A", "length_ft": "120", "current_a": "35"},
    {"name": "Feeder B", "length_ft": "85", "current_a": "22"},
]

# Write controlled input data.
with input_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)

# Load records from disk.
def load_feeders(path):
    if not path.exists():
        raise FileNotFoundError("Missing feeder input file")

    with path.open(newline="") as file:
        records = list(csv.DictReader(file))
    return records

# Validate and convert records.
def prepare_feeders(records):
    prepared = []
    for record in records:
        length = float(record["length_ft"])
        current = float(record["current_a"])

        if length <= 0 or current <= 0:
            raise ValueError("Length and current must be positive")
        prepared.append({"name": record["name"], "length": length, "current": current})
    return prepared

# Calculate voltage drop percent.
def voltage_drop_percent(length_ft, current_a, volts=240):
    resistance_per_foot = 0.000491
    drop = 2 * length_ft * current_a * resistance_per_foot
    return 100 * drop / volts

# Build formatted report lines.
def build_report(feeders):
    lines = ["Voltage Drop Report", "Name       Drop %   Status"]
    for feeder in feeders:
        drop = voltage_drop_percent(feeder["length"], feeder["current"])
        status = "OK" if drop <= 3 else "CHECK"

        line = f"{feeder['name']:<10} {drop:>5.2f}   {status}"
        lines.append(line)
    return lines

# Run the tool workflow.
try:
    raw_records = load_feeders(input_path)
    feeders = prepare_feeders(raw_records)
    report_lines = build_report(feeders)

    report_path.write_text("\n".join(report_lines))
    print("Tool completed successfully.")
    print(f"Input feeders processed: {len(feeders)}")
    print(f"Report saved to: {report_path}")
    print(report_lines[0])

    print(report_lines[1])
    print(report_lines[2])
    print(report_lines[3])
except (FileNotFoundError, ValueError, KeyError) as error:
    print(f"Tool stopped safely: {error}")



### **2.3. Reusable Report Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_02_03.jpg?v=1780966882" width="250">



>* Separate calculations from clear result presentation
>* Create consistent reports from structured data

>* Use consistent result structures for reports
>* Centralize formatting to improve maintenance and testing

>* Connect file data to clear reports
>* Turn calculations into useful communication



In [ ]:
#@title Python Code - Reusable Report Functions

# Reusable reports keep engineering tools maintainable.
# Structured results make output formatting consistent.
# This example summarizes branch voltage drops.

from pathlib import Path
from datetime import datetime

# Define a tiny engineering result dataset.
results = [
    {"circuit": "Panel A", "current_a": 18.0, "length_ft": 85.0,
     "drop_v": 3.8, "limit_v": 4.8},
    {"circuit": "Pump Feed", "current_a": 32.0, "length_ft": 140.0,
     "drop_v": 5.6, "limit_v": 4.8},
    {"circuit": "Lighting", "current_a": 9.0, "length_ft": 60.0,
     "drop_v": 1.4, "limit_v": 4.8},
]

# Add pass or warning status values.
for item in results:
    item["status"] = "PASS" if item["drop_v"] <= item["limit_v"] else "CHECK"

# Create one reusable report formatter.
def build_voltage_report(project_name, records):
    if not records:
        return "No records were available for reporting."

    # Build a readable report as text lines.
    lines = []
    lines.append(f"Voltage Drop Report: {project_name}")
    lines.append(f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    lines.append("Circuit       Current  Length  Drop   Limit  Status")
    lines.append("------------- -------- ------- ------ ------ ------")

    # Format each result consistently.
    for row in records:
        line = (
            f"{row['circuit']:<13} "
            f"{row['current_a']:>6.1f}A "
            f"{row['length_ft']:>5.0f}ft "
            f"{row['drop_v']:>4.1f}V "
            f"{row['limit_v']:>4.1f}V "
            f"{row['status']:>6}"
        )
        lines.append(line)

    # Add a short engineering summary.
    checked = sum(row["status"] == "CHECK" for row in records)
    lines.append(f"Summary: {checked} circuit(s) require review.")
    return "\n".join(lines)

# Generate one report for screen and file use.
report_text = build_voltage_report("Training Lab Feeders", results)
report_path = Path("voltage_drop_report.txt")

# Save the same formatted report to disk.
report_path.write_text(report_text, encoding="utf-8")

# Print a compact confirmation and the report.
print(report_text)
print(f"Saved report file: {report_path.name}")



## **3. Final Tool Evaluation**

### **3.1. Correctness Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_03_01.jpg?v=1780966872" width="250">



>* Test results against trusted known cases
>* Check units, ranges, rounding, and assumptions

>* Test normal, extreme, and invalid inputs
>* Use realistic scenarios to confirm dependable results

>* Fail safely with clear, actionable errors
>* Show inputs, units, warnings for review



In [ ]:
#@title Python Code - Correctness Checks

# Evaluate tool correctness before trusting engineering outputs.
# Use known cases and boundary cases.
# Keep reports traceable, readable, and safe.

# Define a small voltage drop calculator.
def voltage_drop_percent(current_a, length_ft, resistance_ohm_per_1000ft, voltage_v):

    # Reject values that are not physically meaningful.
    if current_a < 0 or length_ft < 0 or voltage_v <= 0:

        return None, "Invalid input: use nonnegative current and length."

    # Apply single phase round trip conductor length.
    drop_v = current_a * (2 * length_ft / 1000) * resistance_ohm_per_1000ft

    # Convert voltage drop into percent.
    percent = 100 * drop_v / voltage_v

    return percent, "OK"

# Store ordinary and boundary test cases.
test_cases = [
    {"name": "known manual case", "amps": 10, "feet": 100, "ohms": 1, "volts": 120, "expected": 1.667},
    {"name": "zero current case", "amps": 0, "feet": 50, "ohms": 1, "volts": 120, "expected": 0.000},
    {"name": "negative current case", "amps": -5, "feet": 50, "ohms": 1, "volts": 120, "expected": None},
]

# Check each case against expected behavior.
results = []
for case in test_cases:

    # Run the engineering calculation safely.
    value, message = voltage_drop_percent(case["amps"], case["feet"], case["ohms"], case["volts"])

    # Compare valid answers using practical rounding tolerance.
    passed = value is None if case["expected"] is None else abs(value - case["expected"]) < 0.01

    # Save a compact traceable result.
    results.append((case["name"], passed, value, message))

# Print a short correctness report.
print("Correctness check report")
print("Tool: single-phase voltage drop estimator")

# Show traceable outcomes without excessive output.
for name, passed, value, message in results:

    # Format numeric and invalid results clearly.
    shown = "invalid" if value is None else f"{value:.3f}%"

    print(f"{name}: {'PASS' if passed else 'FAIL'} | {shown} | {message}")

# Summarize whether the tool is ready for review.
all_passed = all(item[1] for item in results)
print(f"Ready for engineering review: {'YES' if all_passed else 'NO'}")



### **3.2. Readability Review**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_03_02.jpg?v=1780966875" width="250">



>* Readable tools support trust, maintenance, and safety
>* Organized sections make engineering logic easier to review

>* Use engineering terms for clear names
>* Write helpful comments and user messages

>* Organize code for safe, simple updates
>* Format reports for quick, clear interpretation



In [ ]:
#@title Python Code - Readability Review

# Review readability before trusting engineering automation.
# Names should match electrical engineering meaning.
# Reports need clear labels and units.

from pathlib import Path
import json

# Store one small feeder design example.
feeder_case = {
    "feeder_name": "Panel LP-2 feeder",
    "system_voltage_v": 480,

    "load_current_a": 92.4,
    "conductor_rating_a": 125,
    "voltage_drop_percent": 2.7,
    "max_drop_percent": 3.0,

    "report_file": "readability_review_report.txt"
}

# Check required engineering fields early.
required_fields = [
    "feeder_name",
    "system_voltage_v",
    "load_current_a",

    "conductor_rating_a",
    "voltage_drop_percent",
    "max_drop_percent",
    "report_file"
]

# Collect missing fields before calculating.
missing_fields = [
    field for field in required_fields
    if field not in feeder_case
]

# Stop safely if the case is incomplete.
if missing_fields:
    raise ValueError(f"Missing required fields: {missing_fields}")

# Calculate readable engineering results.
load_margin_a = (
    feeder_case["conductor_rating_a"]
    - feeder_case["load_current_a"]
)

# Classify the feeder using clear limits.
loading_ok = load_margin_a >= 0
voltage_drop_ok = (
    feeder_case["voltage_drop_percent"]
    <= feeder_case["max_drop_percent"]
)

# Build report lines with labels and units.
report_lines = [
    "READABILITY REVIEW REPORT",
    f"Feeder: {feeder_case['feeder_name']}",
    f"System voltage: {feeder_case['system_voltage_v']} V",

    f"Load current: {feeder_case['load_current_a']:.1f} A",
    f"Conductor rating: {feeder_case['conductor_rating_a']:.1f} A",
    f"Current margin: {load_margin_a:.1f} A",

    f"Voltage drop: {feeder_case['voltage_drop_percent']:.1f} %",
    f"Maximum allowed drop: {feeder_case['max_drop_percent']:.1f} %",
    f"Loading check: {'PASS' if loading_ok else 'FAIL'}",

    f"Voltage-drop check: {'PASS' if voltage_drop_ok else 'FAIL'}",
    "Review note: names, limits, and units are visible."
]

# Write a small shareable text report.
report_path = Path(feeder_case["report_file"])
report_path.write_text("\n".join(report_lines), encoding="utf-8")

# Save compact structured data for auditing.
summary_path = Path("readability_review_summary.json")
summary_data = {
    "loading_ok": loading_ok,
    "voltage_drop_ok": voltage_drop_ok,
    "load_margin_a": round(load_margin_a, 1)
}

# JSON keeps results machine-readable.
summary_path.write_text(
    json.dumps(summary_data, indent=2),
    encoding="utf-8"
)

# Display concise evidence for learners.
print("Readability review completed.")
print(f"Report file: {report_path.name}")
print(f"Feeder name: {feeder_case['feeder_name']}")
print(f"Current margin: {load_margin_a:.1f} A")
print(f"Loading check: {'PASS' if loading_ok else 'FAIL'}")
print(f"Voltage-drop check: {'PASS' if voltage_drop_ok else 'FAIL'}")



### **3.3. Workflow Usefulness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_B/image_03_03.jpg?v=1780966876" width="250">



>* Fit tools into real engineering workflows
>* Save effort and produce clear results

>* Evaluate tools in realistic decision scenarios
>* Match guidance and reports to user needs

>* Reliable tools support repeated engineering tasks
>* Clear outputs build confidence and adaptability



In [ ]:
#@title Python Code - Workflow Usefulness

# Evaluate workflow usefulness with engineering examples.
# Compare tool effort against manual effort.
# Keep output concise for design review.

# Import built-in file handling helpers.
import csv, io

# Store compact feeder cases as CSV text.
csv_text = "name,volts,amps,length_ft,vd_percent\nFeeder_A,480,120,180,2.7\nFeeder_B,480,95,260,3.4\nFeeder_C,208,60,90,2.1"

# Define workflow scoring thresholds.
manual_minutes = 24

tool_minutes = 6

# Read rows like a small project file.
file_like = io.StringIO(csv_text)

reader = csv.DictReader(file_like)

# Convert each feeder row into useful records.
records = []

for row in reader:
    percent = float(row["vd_percent"])
    status = "PASS" if percent <= 3.0 else "REVIEW"
    records.append({"name": row["name"], "status": status, "vd": percent})

# Check that the workflow has usable data.
if len(records) == 0:
    raise ValueError("No feeder records were loaded")

# Summarize decisions the engineer can act on.
review_count = sum(item["status"] == "REVIEW" for item in records)

saved_minutes = manual_minutes - tool_minutes

# Build short report lines for stakeholders.
report_lines = [
    "Workflow usefulness check for voltage-drop tool",
    f"Feeder cases processed: {len(records)}",
    f"Manual time estimate: {manual_minutes} minutes",
    f"Tool time estimate: {tool_minutes} minutes",
    f"Estimated time saved: {saved_minutes} minutes",
    f"Borderline cases needing review: {review_count}",
]

# Print the report without overwhelming learners.
for line in report_lines:
    print(line)

# Show one actionable engineering result.
worst_case = max(records, key=lambda item: item["vd"])

print(f"Worst case: {worst_case['name']} at {worst_case['vd']}% voltage drop")

print("Useful when outputs guide the next engineering decision.")



# <font color="#418FDE" size="6.5" uppercase>**Capstone Tool Design**</font>


In this lecture, you learned to:
- Design a command-line engineering tool with clear inputs, calculations, outputs, and error behavior. 
- Integrate functions, collections, file handling, and formatted reports into one coherent script. 
- Evaluate the final tool for correctness, readability, and usefulness in an electrical engineering workflow. 

<font color='yellow'>Congratulations on completing this course!</font>